In [1]:
MAKE_TSNE = False

In [ ]:
import pickle
from einops import rearrange
from lightning import LightningDataModule
from typing import Literal
from datasets import load_dataset, load_from_disk, DatasetDict, VerificationMode, Dataset   
import os
from torch.utils.data import DataLoader
import torch
import numpy as np
from tqdm import tqdm

class TSQADataModule(LightningDataModule):
    KEY = "dasyd/quants"

    def __init__(
        self,
        batch_size: int = 32,
        task: Literal["binary", "multi", "open", "count"] = "multi",
    ):
        super().__init__()

        self.batch_size = batch_size
        self.task = task
        self.cache = f"/workspaces/ts-qa/preprocess_data/ts_qa_{self.task}"
        self.action_2_idx = {}

    def load_data(self,splits=["val", "train", "test"]) -> DatasetDict:
        ds = load_dataset(
            TSQADataModule.KEY,
            self.task,
            data_dir=self.task,
            data_files={split: f"{split}-*" for split in splits},
            verification_mode=VerificationMode.NO_CHECKS,
            num_proc=1 #len(splits),
        )
        return ds # type: ignore



    def process(self):
    
        ds = self.load_data()
        
        ds = ds.with_format("torch")

        action_2_idx = {}

        def convert_action(action_dict):
            action_list = action_dict["action"]
            for action in action_list:
                if action not in action_2_idx:
                    action_2_idx[action] = len(action_2_idx)
            return [action_2_idx[action] for action in action_list]
        
        def make_action_split(x):
            traj = x["trajectory"].view(-1, 24*3)
            # split trajectory along the time axis
            traj = rearrange(traj, '(act cnt) channels -> act cnt channels', cnt=80)

            action_ids = convert_action(x["action_sequence"])

            return action_ids, traj


        for sp in ["train", "val", "test"]:
            ds_split = ds[sp]

            trajs = []
            action_ids_lst = []

            for i in tqdm(range(len(ds_split))):
                action_ids, traj = make_action_split(ds_split[i])

                trajs.append(traj)
                action_ids_lst.append(torch.tensor(action_ids,dtype=torch.long))

            trajs = torch.concat(trajs)
            action_ids = torch.concat(action_ids_lst)

            new_ds = Dataset.from_dict( {"action_ids": action_ids, "trajectory": trajs})



                # ds_split[i]["action_ids"] = torch.tensor(action_ids)
                # ds_split[i]["trajectory"] = traj

            # ds_split = ds_split.map(
            #     lambda x: {
            #         "trajectory": x["trajectory"],
            #         "question": x["question"],
            #         "answer": x["answer"],
            #     }
            # )
            # ds_split = ds_split.map(lambda x: {"trajectory": x["trajectory"].view(-1, 24*3)})
            # ds_split = ds_split.map(lambda x: {"action_ids": convert_action(x["action_sequence"]), "trajectory": x["trajectory"].view(-1, 24*3)},remove_columns=["action_sequence"])


            


            ds[sp] = new_ds

        
            # int_to_answer = [
            #         "no",
            #         "yes",
            #         "0",
            #         "1",
            #         "2",
            #         "open the front door",
            #         "clean the table",
            #         "open the third drawer",
            #         "close the front door",
            #         "toggle the switch",
            #         "close the third drawer",
            #         "open the second drawer",
            #         "close the first drawer",
            #         "close the second drawer",
            #         "open the first drawer",
            #         "close the back door",
            #         "open the back door",
            #         "close the fridge",
            #         "open the fridge",
            #         "close the dishwasher",
            #         "drink from the cup",
            #         "open the dishwasher",
            #         "3",
            #         "6",
            #         "4",
            #         "7",
            #     ]


            # def tokenize_and_pad(examples, word_idx, maxlen=31):
            #     # Tokenize the question
            #     tokens = word_tokenize(examples["question"])
                
            #     # Convert tokens to their corresponding indices
            #     tok_indices = [word_idx.get(token.lower(), 0) for token in tokens]
                
            #     # Pad the token indices
            #     padded_tok = tok_indices[:maxlen] + [0] * max(0, maxlen - len(tok_indices))
                
            #     return {'text_idx': padded_tok, "answer_text": int_to_answer[examples["answer"]]}
            
            # Apply the transformation to the dataset
            # tok_data = ds_split.map(lambda x: tokenize_and_pad(x, word_idx=word_idx), batched=False)
            # ds[sp] = tok_data
        
        ds.save_to_disk(self.cache)
        pickle.dump(action_2_idx, open(f"{self.cache}/action2idx.pkl", "wb"))
        # return ds.with_format("torch")

    def prepare_data(self) -> None:
        # self._load_dataset_split(["val", "train"])
        if not os.path.exists(self.cache):
            self.process()

    def setup(self, stage: str) -> None:
        self.dataset: DatasetDict = load_from_disk(self.cache) # type: ignore
        self.dataset: DatasetDict = self.dataset.with_format("torch")

        self.action_2_idx = pickle.load(open(f"{self.cache}/action2idx.pkl", "rb"))

        # self.dataset = self.dataset.map(lambda x: self.tokenizer(x['question'], padding="max_length", max_length=30, truncation=True), batched=True)

        # self.dataset = self.dataset.with_format(type='torch', columns=['input_ids', 'attention_mask', 'trajectory', 'answer'])

        # self.dataset = self.dataset.with_format("torch")

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            # OppQADataset(self.dataset["train"], scaler=Scaler(), tokenizer=self.tokenizer),
            self.dataset["train"], # type: ignore
            batch_size=self.batch_size,
            shuffle=True,
            pin_memory=True,
            # num_workers=100
            num_workers=10
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.dataset["val"], # type: ignore
            batch_size=self.batch_size,
            pin_memory=True,
            num_workers=10
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.dataset["test"], # type: ignore
            batch_size=self.batch_size,
        )

In [3]:
ts_qa = TSQADataModule(task="binary",batch_size=128)

ts_qa.prepare_data()
ts_qa.setup("fit")
train = ts_qa.train_dataloader()
next(iter(train))

# seq_len = 320  
# action_len = 4
# action_count = 4

# fps = 20

# act_len = 4 * 20


# seq_len = int(ts_qa.dataset["val"][0]["trajectory"].shape[0] * 0.5)
# seq_var = ts_qa.dataset["val"][0]["trajectory"].shape[1]


{'action_ids': tensor([10, 16,  0, 14, 10,  0,  5,  1, 12,  9, 12, 11,  2,  6, 12, 18, 13, 15,
         10, 17, 14, 16,  2,  8,  7, 13,  2,  9,  5, 12,  4,  7,  3,  5,  1,  6,
          3, 14,  3, 15,  8,  2,  9,  8, 10, 15, 18,  6, 17, 13,  2, 14,  5,  9,
         11, 12,  1, 17, 14,  9, 17, 13, 11,  9,  7, 15, 15,  8, 16, 10,  2, 11,
          9, 14, 10,  8,  7, 17, 13, 10,  0, 18, 16,  6, 12, 16,  9,  5, 11,  0,
         15,  2,  2,  1,  8,  6, 15, 17, 18, 18, 18,  8,  8, 16, 12,  1, 12,  3,
          3, 18,  7, 10, 15,  3, 17, 15, 14,  7, 13,  7, 13, 14,  0,  4,  9, 16,
          8,  5]),
 'trajectory': tensor([[[ 0.0000e+00, -0.0000e+00,  9.0382e-01,  ...,  2.9540e-01,
            8.7537e-02,  7.7329e-01],
          [ 3.3739e-04, -1.5192e-04,  9.0381e-01,  ...,  2.9507e-01,
            8.5712e-02,  7.7119e-01],
          [ 6.6564e-04, -4.8107e-04,  9.0377e-01,  ...,  2.9585e-01,
            8.4639e-02,  7.7078e-01],
          ...,
          [ 3.7645e-02,  1.5727e-03,  9.0303e-01, 

In [4]:
ts_qa.action_2_idx

{'holding a baby': 0,
 'shaking hands': 1,
 'running': 2,
 'jumping once': 3,
 'punching': 4,
 'golfing (swinging a club)': 5,
 'drinking with the left hand': 6,
 'skipping rope': 7,
 'dancing': 8,
 'waving': 9,
 'playing guitar': 10,
 'bowing': 11,
 'picking something up with both hands': 12,
 'catching a ball': 13,
 'eating with the right hand': 14,
 'kicking a ball': 15,
 'sitting down': 16,
 'throwing a ball': 17,
 'T-posing': 18}

In [5]:
from lightning import LightningModule
from transformers import (
    PatchTSMixerConfig,
    PatchTSMixerForPrediction,
    PatchTSMixerForPretraining,
    TrainingArguments,
)
from torch.utils.data import DataLoader, Dataset
from transformers.models.patchtsmixer.modeling_patchtsmixer import (
    PatchTSMixerModelOutput,
    PatchTSMixerForPredictionOutput,
    PatchTSMixerForPreTrainingOutput,
    PatchTSMixerForTimeSeriesClassification
)
import lightning as L
from aim.pytorch_lightning import AimLogger
import torchmetrics as tm


class TransformerPretrain(LightningModule):
    def __init__(
        self,
        lr: float = 5e-3,
    ):
        super().__init__()
        self.config = PatchTSMixerConfig(
            context_length=80,
            num_input_channels=72,
            patch_length=8,  # seq_len//16,
            patch_stride=8,  # 320//16,
            return_loss=True,
            mode="mix_channel",
            # head_aggregation="use_last",
            # scaling=,
            d_model=40,
            use_positional_encoding=True,
            positional_encoding_type="sincos",
            num_targets=len(ts_qa.action_2_idx),
        )
        self.train_acc = tm.Accuracy(task="multiclass", num_classes=len(ts_qa.action_2_idx))
        self.val_acc = tm.Accuracy(task="multiclass", num_classes=len(ts_qa.action_2_idx))
        self.time_extractor = PatchTSMixerForTimeSeriesClassification(self.config)
        self.lr = lr

    def training_step(self, batch, batch_idx):
        inputs = batch["trajectory"] #[:, :seq_len, :]
        labels = batch["action_ids"]
        # inputs = inputs.transpose(1, 2)  # batch x seq x feat
        outputs: PatchTSMixerForPreTrainingOutput = self.time_extractor(
            inputs, labels,
        )
        self.train_acc(outputs.prediction_outputs, labels)
        loss = outputs.loss
        self.log("train/loss", loss, prog_bar=True, on_epoch=True, on_step=True)
        return loss
    
        

    def validation_step(self, batch, batch_idx):
        inputs = batch["trajectory"] #[:, :seq_len, :]
        labels = batch["action_ids"]
        # inputs = inputs.transpose(1, 2)  # batch x seq x feat
        outputs: PatchTSMixerForPreTrainingOutput = self.time_extractor(
            inputs, labels,
        )
        self.val_acc(outputs.prediction_outputs, labels)
        loss = outputs.loss
        self.log("val/loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def on_train_epoch_end(self) -> None:
        self.log("train/acc", self.train_acc.compute(), prog_bar=True, on_epoch=True)
        self.train_acc.reset()

        self.log("val/acc", self.val_acc.compute(), prog_bar=True, on_epoch=True)
        self.val_acc.reset()


        


    def configure_optimizers(self):
        return torch.optim.RAdam(self.parameters(), lr=self.lr)


transform_model = TransformerPretrain()
logger = AimLogger(experiment="AccSep", log_system_params=False)
trainer = L.Trainer(max_epochs=5, logger=logger)

trainer.fit(transform_model, datamodule=ts_qa)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type                                    | Params | Mode 
-----------------------------------------------------------------------------------
0 | train_acc      | MulticlassAccuracy                      | 0      | train
1 | val_acc        | MulticlassAccuracy                      | 0      | train
2 | time_extractor | PatchTSMixerForTimeSeriesClassification | 167 K  | train
-----------------------------------------------------------------------------------
167 K     Trainable params
400       Non-trainable params
167 K     Total params
0.671     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


In [6]:
# trainer.save_checkpoint("transformer_pretrain_10e_1.030.ckpt")

In [7]:
if MAKE_TSNE:
    sample = ts_qa.dataset["train"][0:10000]
    traj = sample["trajectory"]
    transform_model = transform_model.to("cuda")

    last_hidden_states = []
    transform_model.eval()
    with torch.no_grad():
        for i in range(0, traj.shape[0], 128):
            tr = traj[i:i+128]
            tr = tr.to("cuda")
            out = transform_model.time_extractor.model(tr)
            pass
            last_hidden_states.append(out.last_hidden_state.detach())

    last_hidden_states = torch.cat(last_hidden_states, dim=0)
    res = last_hidden_states.max(1)[0].cpu().detach().numpy()
    # res = torch.mean(last_hidden_states, dim=1).cpu().detach().numpy()





In [8]:
if MAKE_TSNE:
    from sklearn.manifold import TSNE
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap


    features = []
    actions = []

    for i in range(len(res)):
        el = res[i]
        acts = sample["action_ids"][i]
        
        # for j in range(len(acts)):
        feat_el = el.flatten()# el[j*4:(j*4+4)].flatten()  # Flatten the 3 tokens x 256 dimensions into a single vector
        features.append(feat_el)  # Convert PyTorch tensor to numpy array
        actions.append(acts)

    # Convert the features and actions to numpy arrays
    features = np.array(features)
    actions = np.array(actions)


    # Apply t-SNE to reduce dimensions
    tsne = TSNE(n_components=2, random_state=0)
    features_2d = tsne.fit_transform(features)

    # Define a discrete colormap
    act_len = len(ts_qa.action_2_idx)
    cmap = plt.cm.tab20
    cmap = ListedColormap(cmap.colors[:act_len])  # Extract only the required number of colors

    # Create a scatter plot with categorical colors
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=actions, cmap=cmap)

    # Add a legend with action labels
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap(i), markersize=10) for i in range(act_len)]
    labels = ts_qa.action_2_idx.keys()
    plt.legend(handles, labels, title='Actions')

    # Add labels and title
    plt.title('t-SNE of Action Embeddings')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')

    # Show the plot
    plt.show()



In [9]:
ds = load_dataset(
    TSQADataModule.KEY,
    "binary",
    data_dir="binary",
    data_files={split: f"{split}-*" for split in ["val"]},
    verification_mode=VerificationMode.NO_CHECKS,
    num_proc=1,
).with_format("torch")


Generating val split:   0%|          | 0/2080 [00:00<?, ? examples/s]

In [10]:
# create a user prompt with a action db coming from the dict ts_qa.action_2_idx

db = {"role": "user", "content": "\n".join([f"[Action]{action}[\\Action][TimeseriesID]{id}[\\TimeseriesID]" for action, id in ts_qa.action_2_idx.items()])}

transform_model = transform_model.to("cuda")
el = ds["val"][100]
traj = el["trajectory"]
traj = rearrange(traj, '(act cnt) channels a -> act cnt (channels a)', cnt=80)
traj.shape
question = el["question"] 
answer = el["answer_text"]
# question
ids = transform_model.time_extractor(traj.to("cuda")).prediction_outputs.detach().argmax(1).tolist()
# ids
query = {"role": "user", "content": f"[Timeseries]{ids}[\\Timeseries][Question]{question}[\\Question]"}
example_query = {"role": "user", "content": f"[Timeseries]9,8,10,11[\\Timeseries][Question]Is the person waving before dancing?[\\Question]"}
example_answer = {"role": "assistant", "content": "[Answer]Yes[\\Answer]"}

In [11]:
make_db_template = lambda q: [
    # {"role": "system", "content": "You are a multimodal time series question answering model. You are supposed to answer questions about time series data. IDs in the [Timeseries]...[\\Timeseries] tags are the action IDS. You can look up the action IDs in the [Action]...[\\Action] tags to understand the actions in the time series."},
    {
        "role": "system",
        "content": "You are a multimodal time series question answering model designed to analyze sequences of actions and respond to queries about them. Your input will be provided in two specific tags: [Action] and [TimeseriesID]. The [Action] tags describe different activities, each associated with a unique [TimeseriesID]. When answering questions, refer to these IDs to understand the sequence and nature of the actions in the [Timeseries]...[\Timeseries] tags. Responses to questions should be enclosed in [Answer]...[\Answer] tags. If an explanation is necessary, enclose it in [Explanation]...[\Explanation] tags. Your responses should focus on interpreting the action sequences accurately and providing clear answers or explanations as required.",
    },
    db,
    example_query,
    example_answer,
    q,
]

messages = make_db_template(query)

print(messages)

[{'role': 'system', 'content': 'You are a multimodal time series question answering model designed to analyze sequences of actions and respond to queries about them. Your input will be provided in two specific tags: [Action] and [TimeseriesID]. The [Action] tags describe different activities, each associated with a unique [TimeseriesID]. When answering questions, refer to these IDs to understand the sequence and nature of the actions in the [Timeseries]...[\\Timeseries] tags. Responses to questions should be enclosed in [Answer]...[\\Answer] tags. If an explanation is necessary, enclose it in [Explanation]...[\\Explanation] tags. Your responses should focus on interpreting the action sequences accurately and providing clear answers or explanations as required.'}, {'role': 'user', 'content': '[Action]holding a baby[\\Action][TimeseriesID]0[\\TimeseriesID]\n[Action]shaking hands[\\Action][TimeseriesID]1[\\TimeseriesID]\n[Action]running[\\Action][TimeseriesID]2[\\TimeseriesID]\n[Action]ju

In [13]:
import transformers
import torch

from transformers import LlamaForCausalLM, LlamaTokenizer, PreTrainedTokenizerFast, AutoTokenizer, AutoModelForCausalLM
model_path = "/common-repos/LMs/Llama3_converted/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)






input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# print(tokenizer.decode(input_ids[0], skip_special_tokens=True))



terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    eos_token_id=terminators,
    # do_sample=True,
    temperature=0.6,
    # top_p=0.9,
)
response = outputs[0][input_ids.shape[-1]:]
resp_str = tokenizer.decode(response, skip_special_tokens=True)
# parse the response, the answer is between the [Answer]...[\Answer] tags and a potential explanation is between the [Explanation]...[\Explanation] tags
pred_answer = resp_str.split("[Answer]")[1].split("[\\Answer]")[0]
if "[Explanation]" in resp_str:
    explanation = resp_str.split("[Explanation]")[1].split("[\\Explanation]")[0]
else:
    explanation = None

final_str = f"GT:Answer: {answer}\nPred:Answer: {pred_answer}\nExplanation: {explanation}"
print(final_str)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

You are a multimodal time series question answering model designed to analyze sequences of actions and respond to queries about them. Your input will be provided in two specific tags: [Action] and [TimeseriesID]. The [Action] tags describe different activities, each associated with a unique [TimeseriesID]. When answering questions, refer to these IDs to understand the sequence and nature of the actions in the [Timeseries]...[\Timeseries] tags. Responses to questions should be enclosed in [Answer]...[\Answer] tags. If an explanation is necessary, enclose it in [Explanation]...[\Explanation] tags. Your responses should focus on interpreting the action sequences accurately and providing clear answers or explanations as required.user

[Action]holding a baby[\Action][TimeseriesID]0[\TimeseriesID]
[Action]shaking hands[\Action][TimeseriesID]1[\TimeseriesID]
[Action]running[\Action][TimeseriesID]2[\TimeseriesID]
[Action]jumping once[\Action][TimeseriesID]3[\TimeseriesID]
[Action]punch

In [23]:
from torchmetrics.text.bert import BERTScore
bertscore = BERTScore()

bertscore(["The answer is yes"], ["The answer is No"])
# pred_answer, answer

/usr/local/lib/python3.11/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: The argument `model_name_or_path` was not specified while it is required when the default `transformers` model is used. It will use the default recommended model - 'roberta-large'.
  warnings.warn(*args, **kwargs)  # noqa: B028
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': tensor(0.9940), 'recall': tensor(0.9940), 'f1': tensor(0.9940)}

In [25]:
# eval messages
make_eval_template = lambda pred, gt: [
    {
        "role": "system",
        "content": "You are an evaluation model tasked with assessing the similarity of predicted answers compared to reference answers. For each evaluation, present the predicted answer text within the [Prediction] tag and the reference answer text within the [Reference] tag. Determine if the predicted and reference answers are equivalent. Return '1' for equivalent answers and '0' for non-equivalent answers, enclosed in the [Answer] tag. Additionally, provide your assessment of similarity for each pair in the [Similarity] tag, expressed as a percentage. Only the [Answer] and [Similarity] tags should be returned; no further explanation is required.",
    },
    {
        "role": "user",
        "content": f"[Prediction]{pred}[\\Prediction][Reference]{gt}[\\Reference]",
    },
]

eval_messages = make_eval_template(pred_answer, answer)

input_ids = tokenizer.apply_chat_template(
    eval_messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

# print(tokenizer.decode(input_ids[0], skip_special_tokens=True))


terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    eos_token_id=terminators,
    # do_sample=True,
    temperature=0.6,
    # top_p=0.9,
)

response = outputs[0][input_ids.shape[-1] :]
resp_str = tokenizer.decode(response, skip_special_tokens=True)

answer 

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'[Answer]0[/Answer]\n[Similarity]0%[/Similarity]'